# Tratamiento de datos

Antes de hablar de probabilidad necesitamos **describir** los datos: ubicarlos,
cuantificar su dispersiÃ³n y detectar puntos raros. Esta unidad arma el vocabulario
que el resto del libro va a usar como base.

Vamos a llevar tres situaciones en paralelo a lo largo del libro:

- **Hilo A â€” Sala de espera de una clÃ­nica.** Cada paciente que entra registra el
  tiempo (en minutos) que espera hasta ser atendido.
- **Hilo B â€” Encuesta electoral.** Sobre una muestra de personas, cuÃ¡ntas dicen
  preferir cada opciÃ³n.
- **Hilo C â€” LÃ­nea de producciÃ³n.** Cantidad de piezas defectuosas inspeccionadas
  por turno.

En esta unidad nos concentramos en **describir** las observaciones de cada hilo;
todavÃ­a no hablamos de probabilidades.


## Preguntas de inicio

1. Â¿CÃ³mo resumimos en pocos nÃºmeros los tiempos de espera de la clÃ­nica?
2. Â¿CÃ³mo detectamos un dÃ­a raro: un paciente que esperÃ³ muchÃ­simo, un turno con
   muchas piezas defectuosas?
3. Â¿Por quÃ© a veces la **media** miente y conviene mirar la **mediana**?
4. Â¿CÃ³mo comparamos dispersiones de muestras con escalas distintas (minutos vs.
   piezas)?

Vamos a responder cada una al final, en la secciÃ³n **SÃ­ntesis y respuestas**.


## Importaciones

In [ ]:
import numpy as np
import pandas as pd

from core import Observations, Settings
from descriptive import (
    FrequencyTableInput,
    build_frequency_table,
    detect_outliers_tukey,
    standardize_observations,
    summarize_observations,
)
from exercises import NumericAnswerInput, verify_numeric_answer
from visualization import (
    DescriptiveSummaryChartInput,
    FrequencyChartInput,
    HistogramChartInput,
    chart_descriptive_summary,
    chart_frequency_table,
    chart_histogram,
)
from widgets import DescriptiveExplorerInput, build_descriptive_explorer

In [ ]:
settings = Settings()

## Hilo A: una muestra de tiempos de espera

Registramos los minutos que esperaron 80 pacientes en una maÃ±ana en la clÃ­nica.
Para que el ejemplo sea reproducible, los simulamos a partir de un mecanismo
centrado en 4 minutos con cierta variabilidad. La unidad de **probabilidad** que
justifica este mecanismo aparece reciÃ©n en el capÃ­tulo siguiente; por ahora la
tratamos como una muestra concreta.


In [ ]:
rng_clinic = np.random.default_rng(settings.random_seed)
raw_waiting_times = rng_clinic.normal(loc=4.0, scale=1.2, size=80).clip(min=0.0)
waiting_times = Observations.validate(pd.DataFrame({"value": raw_waiting_times}))
waiting_times.head()

Antes de cualquier nÃºmero, **dibujamos**. Construimos una tabla de frecuencias y
la usamos para mostrar dos cosas a la vez: el histograma (barras, frecuencia
absoluta) y la **ojiva** (lÃ­nea, frecuencia relativa acumulada).


In [ ]:
frequency_input = FrequencyTableInput(observations=waiting_times, bin_count=10)
frequency_table = build_frequency_table(frequency_input)

frequency_chart_input = FrequencyChartInput(
    frequency_table=frequency_table,
    title="Tiempos de espera (clÃ­nica) â€” histograma y ojiva",
    settings=settings,
)
chart_frequency_table(frequency_chart_input)

## Resumir con tres miradas: posiciÃ³n, dispersiÃ³n y outliers

La **media muestral** $\bar{x}$ aparece como pregunta natural: Â¿cuÃ¡l es el centro
de gravedad de la muestra? Con $n$ observaciones $x_1, \dots, x_n$:

$$ \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \tag{1.1} $$

La **mediana** $\tilde{x}$ es el valor que parte la muestra ordenada por la mitad.
Para $n$ par se promedia el par central:

$$ \tilde{x} = \begin{cases}
  x_{((n+1)/2)} & \text{si } n \text{ es impar} \\
  \dfrac{x_{(n/2)} + x_{(n/2+1)}}{2} & \text{si } n \text{ es par}
\end{cases} \tag{1.2} $$

El **desvÃ­o estÃ¡ndar muestral** $s$ mide cuÃ¡nto se separan, en promedio, las
observaciones respecto de $\bar{x}$. Se define paso a paso. **Paso 1:** partimos
de la fÃ³rmula (1.1) para tener el centro:

$$ \bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i $$

**Paso 2:** sumamos los desvÃ­os al cuadrado (cuadrar elimina los signos):

$$ \text{SS} = \sum_{i=1}^{n}(x_i - \bar{x})^2 $$

**Paso 3:** dividimos por $n-1$ y tomamos raÃ­z para volver a la unidad original:

$$ s = \sqrt{\frac{\text{SS}}{n - 1}} \tag{1.3} $$


In [ ]:
summary = summarize_observations(waiting_times)
summary

El objeto `summary` tiene tres bloques: **posiciÃ³n** (`location`), **dispersiÃ³n**
(`dispersion`) y **outliers**. Cada bloque concentra los nÃºmeros necesarios para
responder una pregunta distinta: dÃ³nde estÃ¡ el centro, quÃ© tan apretadas son las
esperas, quÃ© pacientes esperaron raro.


## Por quÃ© la mediana resiste lo que la media no

Cuando la distribuciÃ³n es **simÃ©trica**, $\bar{x}$ y $\tilde{x}$ estÃ¡n casi
pegados. Cuando aparecen valores muy altos (un paciente con espera anormal) la
**media se mueve** hacia ese punto; la **mediana se queda quieta** porque sÃ³lo
depende del orden.

Lo materializamos con un boxplot que marca, ademÃ¡s, la media muestral con una
lÃ­nea punteada: cuando hay cola, las dos referencias se separan.


In [ ]:
descriptive_chart_input = DescriptiveSummaryChartInput(
    observations=waiting_times,
    statistics=summary,
    title="Tiempos de espera â€” boxplot con media muestral",
    settings=settings,
)
chart_descriptive_summary(descriptive_chart_input)

## DetecciÃ³n de outliers â€” la regla de Tukey

**Paso 1:** partimos de los cuartiles $Q_1$ y $Q_3$ que ya calculÃ³ la fÃ³rmula
(1.2) en su versiÃ³n generalizada. **Paso 2:** definimos el rango intercuartil
$\text{IQR} = Q_3 - Q_1$. **Paso 3:** una observaciÃ³n es **outlier** si cae
fuera del intervalo:

$$ \bigl[Q_1 - 1{,}5\,\text{IQR},\ Q_3 + 1{,}5\,\text{IQR}\bigr] \tag{1.4} $$

Es la misma regla que usa la caja del boxplot: por eso ese grÃ¡fico es buen
detector visual.


In [ ]:
outlier_report = detect_outliers_tukey(waiting_times)
outlier_report

## PosiciÃ³n relativa: el $z$-score

El $z$-score traduce una observaciÃ³n a la pregunta Â«Â¿a cuÃ¡ntos desvÃ­os del
promedio estÃ¡?Â». **Paso 1:** partimos de las fÃ³rmulas (1.1) y (1.3) para tener
$\bar{x}$ y $s$. **Paso 2:** centramos restando el promedio. **Paso 3:**
reescalamos dividiendo por $s$:

$$ z_i = \frac{x_i - \bar{x}}{s} \tag{1.5} $$

Esta misma transformaciÃ³n va a reaparecer en el capÃ­tulo de variables
aleatorias (ecuaciÃ³n 3.1), pero allÃ¡ con un significado mÃ¡s fuerte: convertir
una Normal cualquiera en la Normal estÃ¡ndar.


In [ ]:
standardized = standardize_observations(waiting_times)
standardized

## Hilo C: piezas defectuosas por turno

Cambiamos de escala y de unidad. La lÃ­nea de producciÃ³n registra, en cada uno
de los Ãºltimos 60 turnos, cuÃ¡ntas piezas resultaron defectuosas tras la
inspecciÃ³n. AcÃ¡ no estamos midiendo en minutos: estamos contando.


In [ ]:
rng_factory = np.random.default_rng(seed=20260202)
raw_defect_counts = rng_factory.poisson(lam=3.5, size=60).astype(float)
defect_counts = Observations.validate(pd.DataFrame({"value": raw_defect_counts}))

defect_histogram_input = HistogramChartInput(
    observations=defect_counts,
    bin_count=12,
    title="Piezas defectuosas por turno (60 turnos)",
    settings=settings,
)
chart_histogram(defect_histogram_input)

In [ ]:
defect_summary = summarize_observations(defect_counts)
defect_summary

Mirando los dos resÃºmenes (clÃ­nica y fÃ¡brica) vemos por quÃ© la **media sola**
no alcanza para comparar: las unidades son distintas (minutos vs. piezas) y la
magnitud tambiÃ©n. Por eso los $z$-scores y el coeficiente de variaciÃ³n
(`dispersion.coefficient_of_variation`) son tan Ãºtiles: son adimensionales.


## ExploraciÃ³n interactiva

El widget genera muestras Normales sintÃ©ticas con $\mu$, $\sigma$ y $n$ a
elecciÃ³n. Sirve para construir intuiciÃ³n: subÃ­ $\sigma$ y mirÃ¡ cÃ³mo la caja del
boxplot se ensancha sin que la mediana se mueva.


In [ ]:
explorer_input = DescriptiveExplorerInput(settings=settings)
build_descriptive_explorer(explorer_input)

## Ejercicio 1 â€” Media de una muestra pequeÃ±a

TomÃ¡ la muestra $\{2, 4, 4, 4, 5, 5, 7, 9\}$ y calculÃ¡ la media muestral
aplicando la fÃ³rmula (1.1).

**Idea.** Sumar y dividir por $n = 8$:

$$ \bar{x} = \frac{2 + 4 + 4 + 4 + 5 + 5 + 7 + 9}{8} = \frac{40}{8} = 5 $$


In [ ]:
exercise_sample = Observations.validate(pd.DataFrame({"value": [2.0, 4.0, 4.0, 4.0, 5.0, 5.0, 7.0, 9.0]}))
expected_mean = summarize_observations(exercise_sample).location.mean

student_answer_mean = 5.0
verify_input_mean = NumericAnswerInput(
    student_answer=student_answer_mean,
    expected_answer=expected_mean,
)
verify_numeric_answer(verify_input_mean)

## Ejercicio 2 â€” DesvÃ­o estÃ¡ndar muestral

Para la misma muestra, aplicÃ¡ la fÃ³rmula (1.3). Con $\bar{x} = 5$ del
ejercicio 1:

$$ \begin{aligned}
\text{SS} &= (2-5)^2 + 3(4-5)^2 + 2(5-5)^2 + (7-5)^2 + (9-5)^2 \\
           &= 9 + 3 + 0 + 4 + 16 = 32 \\
s          &= \sqrt{\dfrac{32}{7}} \approx 2{,}138
\end{aligned} $$


In [ ]:
expected_std = summarize_observations(exercise_sample).dispersion.sample_standard_deviation

student_answer_std = 2.138
verify_input_std = NumericAnswerInput(
    student_answer=student_answer_std,
    expected_answer=expected_std,
    absolute_tolerance=1e-2,
    relative_tolerance=1e-2,
)
verify_numeric_answer(verify_input_std)

## SÃ­ntesis y respuestas

1. **Â¿CÃ³mo resumimos los tiempos de espera?** Con tres bloques: posiciÃ³n
   (`mean`, `median`), dispersiÃ³n (`sample_standard_deviation`,
   `interquartile_range`) y reporte de outliers. La estructura `summary` los
   empaqueta en un Ãºnico objeto.
2. **Â¿CÃ³mo detectamos un dÃ­a raro?** Aplicando la regla de Tukey de la
   ecuaciÃ³n (1.4). Cualquier observaciÃ³n fuera de
   $[Q_1 - 1{,}5\,\text{IQR},\ Q_3 + 1{,}5\,\text{IQR}]$ es candidata a outlier.
3. **Â¿CuÃ¡ndo miente la media?** Cuando la muestra tiene cola larga o un valor
   extremo. La mediana, definida en (1.2), es robusta porque depende del
   orden y no de los valores.
4. **Â¿CÃ³mo comparamos dispersiones de escalas distintas?** Pasando todo a
   $z$-scores (ecuaciÃ³n 1.5) o usando el **coeficiente de variaciÃ³n** $s/\bar{x}$,
   ambos adimensionales.


## PrÃ³ximas preguntas

Hasta acÃ¡ describimos lo que **ya pasÃ³** en la clÃ­nica y en la fÃ¡brica. Las
preguntas que abren el prÃ³ximo capÃ­tulo son:

- Si entra un nuevo paciente, Â¿cuÃ¡l es la **probabilidad** de que espere mÃ¡s
  de 5 minutos?
- Y si ya esperÃ³ 3 minutos, Â¿cÃ³mo cambia esa probabilidad?
- En la encuesta del Hilo B, una persona dice Â«sÃ­Â»: Â¿quÃ© tan creÃ­ble es ese
  resultado dado lo que sabemos de la poblaciÃ³n?

Esas preguntas necesitan **probabilidad** â€” el lenguaje del prÃ³ximo capÃ­tulo.
